# Credit VaR for a Bond Portfolio (CreditMetrics)

*Nova SBE, Credit Risk (2026).*

One-year 99% credit VaR in the CreditMetrics framework: each bond is revalued in every rating state it can migrate to, and credit VaR is the gap between the mean portfolio value and its 1st percentile. $100 is invested in each bond.

1. **Two bonds, analytical:** joint migration probabilities from a bivariate normal with asset correlation 0.3 (the CreditMetrics textbook example) and 0.072.
2. **Two bonds, simulated:** latent asset returns from a three-factor model, 100,000 scenarios.
3. **Full portfolio:** all 17 bonds, constant recovery.
4. **Full portfolio, random recovery:** recovery in default drawn from a Beta distribution.

The inputs (forward zero curves by rating, the one-year transition matrix and the bond list with factor loadings) are course material and not included. To run the notebook, place `CreditVaR_data.xlsx` next to it.

In [1]:
import numpy as np
import pandas as pd

from quantfin.creditmetrics import (
    Bond,
    Portfolio,
    beta_parameters,
    idiosyncratic_std,
    load_workbook,
    simulate_values,
    simulated_credit_var,
    state_bounds,
)

N_SCENARIOS = 100_000
SEED = 42
NOTIONAL = 100


def summary(mean: float, v_star: float, credit_var: float) -> pd.DataFrame:
    return pd.DataFrame(
        {"Value": [mean, v_star, credit_var]},
        index=["Mean portfolio value (mu_V)", "99% critical value (V*)", "Credit VaR (mu_V - V*)"],
    )

## Inputs

Three sheets: one-year forward zero curves per rating, the one-year rating transition matrix, and the bonds (coupon, maturity, rating, recovery mean and standard deviation, loadings on the three factors T, L and C). The factors are standard normal with the correlation matrix below.

In [2]:
FACTOR_CORR = np.array([[1.0, 0.3, 0.1], [0.3, 1.0, 0.2], [0.1, 0.2, 1.0]])  # T, L, C
data = load_workbook("CreditVaR_data.xlsx", FACTOR_CORR)

display(data.forward_curves.round(4), data.transitions.round(4), data.bonds.round(4))
display(pd.DataFrame(FACTOR_CORR, index=["T", "L", "C"], columns=["T", "L", "C"]))

Category,Year 1,Year 2,Year 3,Year 4
One-year forward zero curves (Tab 4.2),,,,
AAA,0.0360,0.0417,0.0473,0.0512
AA,0.0365,0.0422,0.0478,0.0517
A,0.0372,0.0432,0.0493,0.0532
BBB,0.0410,0.0467,0.0525,0.0563
BB,0.0555,0.0602,0.0678,0.0727
B,0.0605,0.0702,0.0803,0.0852
CCC,0.1505,0.1502,0.1403,0.1352


Initial Rating,AAA,AA,A,BBB,BB,B,CCC,Default
One-year transition matrix (tab 4.1),,,,,,,,
AAA,0.9081,0.0833,0.0068,0.0006,0.0012,0.0000,0.0000,0.0000
AA,0.0070,0.9065,0.0779,0.0064,0.0006,0.0014,0.0002,0.0000
A,0.0009,0.0227,0.9105,0.0552,0.0074,0.0026,0.0001,0.0006
BBB,0.0002,0.0033,0.0595,0.8693,0.0530,0.0117,0.0012,0.0018
BB,0.0003,0.0014,0.0067,0.0773,0.8053,0.0884,0.0100,0.0106
B,0.0001,0.0011,0.0024,0.0043,0.0648,0.8346,0.0407,0.0520
CCC,0.0021,0.0000,0.0022,0.0130,0.0238,0.1124,0.6486,0.1979


,CR,Maturity,Rating,R_mean,R_std,Beta_T,Beta_L,Beta_C
issue_id,,,,,,,,
1,0.06,5.0,BBB,0.5113,0.26,0.4,0.2,0.0
2,0.05,3.0,A,0.5113,0.26,0.0,0.0,0.9
3,0.05,2.0,AA,0.5600,0.24,0.5,0.3,0.0
4,0.03,1.0,AAA,0.5600,0.24,0.1,0.4,0.3
5,0.05,5.0,A,0.5600,0.24,0.5,0.2,0.1
6,0.08,3.0,B,0.3500,0.22,0.0,0.9,0.0
7,0.08,1.0,B,0.3500,0.25,0.4,0.2,0.0
8,0.09,2.0,CCC,0.3500,0.22,0.6,0.1,0.0
9,0.06,2.0,BBB,0.4200,0.25,0.3,0.0,0.5


,T,L,C
T,1.0,0.3,0.1
L,0.3,1.0,0.2
C,0.1,0.2,1.0


## Bond and portfolio classes

Both live in [`quantfin/creditmetrics.py`](../src/quantfin/creditmetrics.py).

- **`Bond`** values the bond at the one-year horizon in each rating state by discounting the remaining cash flows on that rating's forward curve; in default it is worth its recovery rate. With the issuer's transition probabilities this gives the expected value and the percentile value.
- **`state_bounds`** maps each rating state to an interval of a standard normal asset return, with thresholds from the cumulative transition probabilities read from default upwards.
- **`Portfolio`** combines two bonds. The joint probability of two states is the bivariate normal mass of the rectangle formed by their intervals.
- **`simulate_values`** draws asset returns from a three-factor model for any number of bonds, maps them to rating states and revalues, optionally with Beta-distributed recovery.

## 1. Two bonds, analytical

### 1.1 Stand-alone bonds

Bond 1: BBB, 6% annual coupon, 5 years. Bond 2: A, 5% coupon, 3 years. Both recover 51.13% in default.

In [3]:
bond_bbb = Bond.from_data(data, 1)
bond_a = Bond.from_data(data, 2)

standalone = pd.DataFrame(
    {
        "Expected value (t+1)": [b.expected_value() * NOTIONAL for b in (bond_bbb, bond_a)],
        "Critical value (V*)": [b.critical_value() * NOTIONAL for b in (bond_bbb, bond_a)],
        "Credit VaR (E[V] - V*)": [b.credit_var() * NOTIONAL for b in (bond_bbb, bond_a)],
    },
    index=["BBB", "A"],
)
display(standalone.round(4))

print("\nTransition probabilities (%):")
display(
    pd.concat([bond_bbb.transition_probs, bond_a.transition_probs], axis=1, keys=["BBB", "A"]) * 100
)

,Expected value (t+1),Critical value (V*),Credit VaR (E[V] - V*)
BBB,107.0694,83.6258,23.4436
A,106.2014,101.3915,4.8099



Transition probabilities (%):


,BBB,A
Initial Rating,,
AAA,0.02,0.09
AA,0.33,2.27
A,5.95,91.05
BBB,86.93,5.52
BB,5.30,0.74
B,1.17,0.26
CCC,0.12,0.01
Default,0.18,0.06


### 1.2 Two-bond portfolio, correlation 0.3

This reproduces the CreditMetrics textbook example. The 1st percentile is read conservatively as the last portfolio value before the cumulative probability reaches 1%.

In [4]:
portfolio = Portfolio((bond_bbb, bond_a), correlation=0.3)

print("Rating thresholds in standard normal space (lower, upper]:")
display(
    pd.DataFrame(
        {"BBB": state_bounds(bond_bbb.transition_probs), "A": state_bounds(bond_a.transition_probs)}
    )
)

print("Value matrix ($100 in each bond)")
display(portfolio.value_matrix().round(2))

print("\nJoint migration probabilities (%)")
display((portfolio.joint_prob_matrix() * 100).round(4))

print("\nDistribution of portfolio values")
display(portfolio.distribution().sort_values("Cum_Prob", ascending=False).round(4))

print("\nPortfolio summary")
display(
    summary(portfolio.mean_value(), portfolio.critical_value(), portfolio.credit_var()).round(4)
)

Rating thresholds in standard normal space (lower, upper]:


,BBB,A
AAA,"(3.540084, inf)","(3.121389, inf)"
AA,"(2.696844, 3.540084)","(1.984501, 3.121389)"
A,"(1.530068, 2.696844)","(-1.507042, 1.984501)"
BBB,"(-1.493142, 1.530068)","(-2.300852, -1.507042)"
BB,"(-2.178081, -1.493142)","(-2.716381, -2.300852)"
B,"(-2.747781, -2.178081)","(-3.194651, -2.716381)"
CCC,"(-2.911238, -2.747781)","(-3.23888, -3.194651)"
Default,"(-inf, -2.911238)","(-inf, -3.23888)"


Value matrix ($100 in each bond)


,AAA,AA,A,BBB,BB,B,CCC,Default
AAA,215.94,215.85,215.66,215.00,212.50,210.74,198.07,160.48
AA,215.76,215.67,215.48,214.82,212.32,210.56,197.89,160.30
A,215.23,215.14,214.95,214.29,211.79,210.03,197.36,159.77
BBB,214.12,214.02,213.84,213.17,210.68,208.92,196.24,158.66
BB,208.59,208.50,208.31,207.65,205.16,203.40,190.72,153.14
B,204.67,204.58,204.39,203.73,201.24,199.48,186.80,149.22
CCC,190.21,190.12,189.93,189.27,186.78,185.02,172.34,134.76
Default,157.72,157.62,157.43,156.77,154.28,152.52,139.84,102.26



Joint migration probabilities (%)


,AAA,AA,A,BBB,BB,B,CCC,Default
AAA,0.0004,0.0034,0.0162,0.0001,0.0000,0.0000,0.0000,0.0000
AA,0.0032,0.0382,0.2866,0.0019,0.0001,0.0000,0.0000,0.0000
A,0.0231,0.3936,5.4440,0.0813,0.0063,0.0015,0.0000,0.0002
BBB,0.0629,1.8097,79.6914,4.5529,0.5740,0.1916,0.0071,0.0405
BB,0.0004,0.0222,4.4650,0.6432,0.1106,0.0446,0.0019,0.0122
B,0.0000,0.0027,0.9276,0.1825,0.0358,0.0158,0.0007,0.0049
CCC,0.0000,0.0002,0.0905,0.0217,0.0046,0.0022,0.0001,0.0007
Default,0.0000,0.0002,0.1287,0.0366,0.0085,0.0042,0.0002,0.0016



Distribution of portfolio values


,State1,State2,Value,Probability,Cum_Prob
63,AAA,AAA,215.9410,0.0000,1.0000
62,AAA,AA,215.8458,0.0000,1.0000
61,AA,AAA,215.7604,0.0000,1.0000
60,AA,AA,215.6653,0.0004,0.9999
59,AAA,A,215.6573,0.0002,0.9995
...,...,...,...,...,...
4,Default,B,152.5215,0.0000,0.0001
3,B,Default,149.2159,0.0000,0.0001
2,Default,CCC,139.8434,0.0000,0.0000
1,CCC,Default,134.7558,0.0000,0.0000



Portfolio summary


,Value
Mean portfolio value (mu_V),213.2708
99% critical value (V*),203.7286
Credit VaR (mu_V - V*),9.5423


### 1.3 Lower correlation: 0.072

The mean is unchanged, since it does not depend on correlation. At this resolution the critical value lands on the same joint state (B, BBB), so credit VaR is the same as in 1.2.

In [5]:
portfolio_low = Portfolio((bond_bbb, bond_a), correlation=0.072)

dist_low = portfolio_low.distribution()
v_star_low = portfolio_low.critical_value()

display((portfolio_low.joint_prob_matrix() * 100).round(4))
display(summary(portfolio_low.mean_value(), v_star_low, portfolio_low.credit_var()).round(4))
display(dist_low[dist_low["Value"] == v_star_low].round(6))

,AAA,AA,A,BBB,BB,B,CCC,Default
AAA,0.0000,0.0008,0.0184,0.0006,0.0001,0.0000,0.0000,0.0000
AA,0.0006,0.0119,0.3035,0.0120,0.0014,0.0004,0.0000,0.0001
A,0.0083,0.1823,5.4639,0.2523,0.0307,0.0101,0.0004,0.0021
BBB,0.0774,1.9659,79.1838,4.7806,0.6386,0.2237,0.0086,0.0514
BB,0.0030,0.0878,4.7678,0.3637,0.0527,0.0194,0.0008,0.0048
B,0.0006,0.0173,1.0462,0.0866,0.0129,0.0049,0.0002,0.0012
CCC,0.0001,0.0016,0.1068,0.0093,0.0014,0.0005,0.0000,0.0001
Default,0.0001,0.0023,0.1596,0.0146,0.0023,0.0009,0.0000,0.0002


,Value
Mean portfolio value (mu_V),213.2708
99% critical value (V*),203.7286
Credit VaR (mu_V - V*),9.5423


,State1,State2,Value,Probability,Cum_Prob
31,B,BBB,203.728556,0.000866,0.004935


## 2. Two bonds, simulated with a factor model

Standardised asset returns load on three correlated factors plus an idiosyncratic shock:

$$r_{\text{Viv}} = 0.4\,T + 0.2\,L + \varepsilon_{\text{Viv}}, \qquad r_{\text{GM}} = 0.9\,C + \varepsilon_{\text{GM}}.$$

Because returns have unit variance, $\sigma^2_\varepsilon = 1 - \beta'\Sigma\beta$. Simulate 100,000 scenarios (the shocks for all bonds are drawn together, as in sections 3 and 4), map each return to a rating state with the thresholds from section 1, revalue, and read the 1st percentile off the simulated distribution.

In [6]:
two_bonds = [bond_bbb, bond_a]
loadings = np.array([b.loadings for b in two_bonds])
eps_std = idiosyncratic_std(loadings, FACTOR_CORR)

variance = pd.DataFrame(
    {
        "Systematic var (beta' Sigma beta)": 1 - eps_std**2,
        "Idiosyncratic var": eps_std**2,
        "Idiosyncratic std": eps_std,
    },
    index=["Vivendi (BBB)", "GM (A)"],
)
print("Variance decomposition")
display(variance.round(4))

values_two = simulate_values(two_bonds, FACTOR_CORR, N_SCENARIOS, SEED, notional=NOTIONAL)
print("\nSimulation summary (two bonds)")
display(summary(*simulated_credit_var(values_two)).round(4))

Variance decomposition


,Systematic var (beta' Sigma beta),Idiosyncratic var,Idiosyncratic std
Vivendi (BBB),0.248,0.752,0.8672
GM (A),0.810,0.190,0.4359



Simulation summary (two bonds)


,Value
Mean portfolio value (mu_V),213.2838
99% critical value (V*),204.3903
Credit VaR (mu_V - V*),8.8935


## 3. Full portfolio, constant recovery

All 17 bonds, each with its own factor loadings, and recovery fixed at the mean recovery rate.

In [7]:
all_bonds = [Bond.from_data(data, int(i)) for i in data.bonds.index]
loadings_all = np.array([b.loadings for b in all_bonds])
eps_std_all = idiosyncratic_std(loadings_all, FACTOR_CORR)

bond_params = pd.DataFrame(
    {
        "Systematic var": 1 - eps_std_all**2,
        "Idiosyncratic std": eps_std_all,
        "R_mean": [b.recovery for b in all_bonds],
        "R_std": [b.recovery_std for b in all_bonds],
    },
    index=data.bonds.index,
)
print("Bond variance decomposition")
display(bond_params.round(4))

values_fixed = simulate_values(all_bonds, FACTOR_CORR, N_SCENARIOS, SEED, notional=NOTIONAL)
print("\nSimulation summary (17 bonds, constant recovery)")
display(summary(*simulated_credit_var(values_fixed)).round(4))

Bond variance decomposition


,Systematic var,Idiosyncratic std,R_mean,R_std
issue_id,,,,
1,0.248,0.8672,0.5113,0.26
2,0.810,0.4359,0.5113,0.26
3,0.430,0.7550,0.5600,0.24
4,0.338,0.8136,0.5600,0.24
5,0.378,0.7887,0.5600,0.24
6,0.810,0.4359,0.3500,0.22
7,0.248,0.8672,0.3500,0.25
8,0.406,0.7707,0.3500,0.22
9,0.370,0.7937,0.4200,0.25



Simulation summary (17 bonds, constant recovery)


,Value
Mean portfolio value (mu_V),1744.4057
99% critical value (V*),1483.4266
Credit VaR (mu_V - V*),260.9790


## 4. Full portfolio, random recovery

Same simulation, but a bond in default recovers a Beta-distributed fraction of par. The Beta parameters follow from the recovery mean $m$ and standard deviation $s$: with $\nu = m(1-m)/s^2 - 1$, $\alpha = m\nu$ and $\beta = (1-m)\nu$. Recovery uncertainty fattens the left tail, so credit VaR rises.

In [8]:
beta_params = pd.DataFrame(
    [beta_parameters(b.recovery, b.recovery_std) for b in all_bonds],
    index=data.bonds.index,
    columns=["Beta alpha", "Beta beta"],
)
print("Recovery distributions")
display(beta_params.round(4))

values_beta = simulate_values(
    all_bonds, FACTOR_CORR, N_SCENARIOS, SEED, notional=NOTIONAL, random_recovery=True
)
print("\nSimulation summary (17 bonds, Beta recovery)")
display(summary(*simulated_credit_var(values_beta)).round(4))

Recovery distributions


,Beta alpha,Beta beta
issue_id,,
1,1.3786,1.3177
2,1.3786,1.3177
3,1.8356,1.4422
4,1.8356,1.4422
5,1.8356,1.4422
6,1.2951,2.4053
7,0.9240,1.7160
8,1.2951,2.4053
9,1.2170,1.6806



Simulation summary (17 bonds, Beta recovery)


,Value
Mean portfolio value (mu_V),1744.3535
99% critical value (V*),1452.7498
Credit VaR (mu_V - V*),291.6037
